In [2]:
import os
import json
from contextlib import AsyncExitStack

from dotenv import load_dotenv
from openai import OpenAI

# Trust the operating system's certificates (helps behind some corporate networks).
import truststore
truststore.inject_into_ssl()

# The two pieces of the MCP client library we need:
#   ClientSession          -> the MCP conversation (initialize / list_tools / call_tool)
#   streamable_http_client -> the HTTP connection that carries it
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

# Load the OpenAI API key.
load_dotenv("/Users/shivam13juna/Documents/scaler/GEN_AI_REF/openai_key.env")
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found. Put OPENAI_API_KEY=sk-... in your .env file.")



MODEL = "gpt-5-nano"                            # cheap + fast, good for tool-use demos
SQL_SERVER_URL = "http://127.0.0.1:8005/mcp"    # the server you started in a terminal

print("Setup done. Model:", MODEL)

Setup done. Model: gpt-5-nano


In [3]:
class MCPHost:
    """
    The MCP HOST for our SQL agent: owns the LLM and lets it use the database tools
    from a running MCP server.

    As in the simpler demo, the app that owns the LLM is the host, and the
    per-server "client" role is just a few methods inside it — not a class of its own.
    """

    def __init__(self, model):
        self.model = model
        self.llm = OpenAI()                 # the language model client
        self.stack = AsyncExitStack()       # holds the open connection; closed at the end

        self.tools = []                     # every tool, described in OpenAI's format
        self.tool_session = {}              # tool name -> the MCP session that runs it

    async def connect(self, name, url):
        """Connect to ONE running MCP server and remember the tools it offers."""
        print(f"Connecting to '{name}' server at {url}")

        # Open the HTTP connection, then start an MCP session on top of it.
        read, write, _ = await self.stack.enter_async_context(streamable_http_client(url))
        session = await self.stack.enter_async_context(ClientSession(read, write))
        await session.initialize()          # the MCP handshake

        # Ask the server what tools it has, and register each one.
        tools_response = await session.list_tools()
        for tool in tools_response.tools:
            self.tools.append({
                "type": "function",
                "name": tool.name,
                "description": tool.description or "",
                "parameters": tool.inputSchema,   # JSON Schema: tells the model the arguments
            })
            self.tool_session[tool.name] = session
            print(f"   found tool: {tool.name}")

    async def call_tool(self, name, arguments):
        """Run a tool and return its text result."""
        session = self.tool_session[name]
        result = await session.call_tool(name, arguments)
        # A tool that returns a list (e.g. list_tables) comes back as several text
        # parts — join them into one string for the model to read.
        return "\n".join(part.text for part in result.content)

    async def ask(self, question):
        """
        Ask a question. The model decides which tools to call and in what order; we
        run them, feed the results back, and loop until it gives a final answer.
        """
        conversation = [{"role": "user", "content": question}]

        while True:
            response = self.llm.responses.create(
                model=self.model,
                instructions=(
                    "You answer questions about a SQLite database using ONLY the tools. "
                    "For each question: (1) call list_tables, (2) call get_table_schema "
                    "on the relevant table(s), (3) write one SQLite SELECT query, "
                    "(4) run it with execute_sql. Never guess table or column names — "
                    "confirm them with the tools first. Then give a short, plain answer."
                ),
                input=conversation,
                tools=self.tools,
            )

            # Did the model ask to call any tools this turn?
            tool_calls = [item for item in response.output if item.type == "function_call"]
            if not tool_calls:
                return response.output_text         # no tools -> this is the final answer

            for call in tool_calls:
                arguments = json.loads(call.arguments)   # model sends args as a JSON string
                print(f"   model wants: {call.name}({arguments})")

                result = await self.call_tool(call.name, arguments)

                # Keep the trace readable: show the result on one line, shortened if long.
                oneline = " ".join(result.split())
                print(f"      tool says: {oneline[:90]}{'...' if len(oneline) > 90 else ''}")

                # Record the call and its result so the model can use them next turn.
                conversation.append({
                    "type": "function_call",
                    "call_id": call.call_id,
                    "name": call.name,
                    "arguments": call.arguments,
                })
                conversation.append({
                    "type": "function_call_output",
                    "call_id": call.call_id,
                    "output": result,
                })

    async def close(self):
        """Close the server connection."""
        await self.stack.aclose()

In [4]:
host = MCPHost(model=MODEL)
await host.connect("sql", SQL_SERVER_URL)

print("\nTools the host can use now:")
for tool in host.tools:
    print(f"  - {tool['name']}: {tool['description']}")

print("\n1) list_tables():")
print(await host.call_tool("list_tables", {}))

print("\n2) get_table_schema('customers'):")
print(await host.call_tool("get_table_schema", {"table_name": "customers"}))

print("\n3) execute_sql(... count EUR customers ...):")
print(await host.call_tool(
    "execute_sql",
    {"query": "SELECT COUNT(*) AS eur_customers FROM customers WHERE currency = 'EUR'"},
))

print("\n4) a blocked write (execute_sql refuses anything but SELECT):")
print(await host.call_tool("execute_sql", {"query": "DELETE FROM customers"}))

# Close in this same cell.
await host.close()
print("\nDone — connection closed.")

Connecting to 'sql' server at http://127.0.0.1:8005/mcp
   found tool: list_tables
   found tool: get_table_schema
   found tool: execute_sql

Tools the host can use now:
  - list_tables: Get all table names in the database. Call this FIRST to discover what tables exist before writing any query.
  - get_table_schema: Get the column names and types for one table. Use this after list_tables, before writing SQL, so you only reference columns that really exist.
  - execute_sql: Run a read-only SQL SELECT query and return the matching rows as text. ONLY SELECT is allowed — no INSERT, UPDATE, DELETE, DROP, or ALTER.

1) list_tables():
customers
employees
orders

2) get_table_schema('customers'):
{
  "name": "id",
  "type": "INTEGER",
  "nullable": true,
  "primary_key": true
}
{
  "name": "name",
  "type": "TEXT",
  "nullable": false,
  "primary_key": false
}
{
  "name": "country",
  "type": "TEXT",
  "nullable": false,
  "primary_key": false
}
{
  "name": "currency",
  "type": "TEXT",
  "nu

In [5]:
host = MCPHost(model=MODEL)
await host.connect("sql", SQL_SERVER_URL)

try:
    questions = [
        "How many customers pay in EUR?",
        "What is the average salary in each department?",
        "Which products were ordered by customers in Germany?",
    ]
    for question in questions:
        print("\n" + "=" * 72)
        print("Q:", question)
        answer = await host.ask(question)
        print("\nA:", answer)
finally:
    # Always close, even if a question errors — still in this same cell.
    await host.close()

Connecting to 'sql' server at http://127.0.0.1:8005/mcp
   found tool: list_tables
   found tool: get_table_schema
   found tool: execute_sql

Q: How many customers pay in EUR?
   model wants: list_tables({})
      tool says: customers employees orders
   model wants: get_table_schema({'table_name': 'customers'})
      tool says: { "name": "id", "type": "INTEGER", "nullable": true, "primary_key": true } { "name": "name...
   model wants: get_table_schema({'table_name': 'orders'})
      tool says: { "name": "id", "type": "INTEGER", "nullable": true, "primary_key": true } { "name": "cust...
   model wants: execute_sql({'query': "SELECT COUNT(*) AS eur_customers FROM customers WHERE currency = 'EUR';"})
      tool says: eur_customers ------------- 4

A: There are 4 customers whose currency is EUR.

Q: What is the average salary in each department?
   model wants: list_tables({})
      tool says: customers employees orders
   model wants: get_table_schema({'table_name': 'employees'})
     

Suppose due to network failure, MCP server becomes unavailable while an executing a flow, how should the client handle partial execution?


# LangChain SDK

In [6]:
%pip install langchain-mcp-adapters


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient
from IPython.display import Image, display

# The question this notebook asks of every agent it builds.
BUSINESS_QUESTION = "What was our total revenue, excluding cancelled orders?"

# One instruction string, shared by all three frameworks, so the comparison in P9 is fair.
# The tool ORDER is spelled out deliberately: left to itself the model will guess a plausible
# table name, query it, and report that the data is missing. Naming the discovery steps costs
# one sentence and removes that whole failure mode.
AGENT_INSTRUCTIONS = (
    "You are InsightAgent, a data analyst for an online-retail store. "
    "Always work in this order: first list the tables, then inspect the schema of every table "
    "you intend to use, and only then write SQL. Never guess a table or column name. "
    "Revenue must EXCLUDE cancelled invoices (invoices.is_cancelled = 1). "
    "State the final answer clearly, including the number."
)

# LangChain's MCP client. Same server the OpenAI SDK cell below uses —
# `transport` + `url` is all it needs (no command, no args).
client = MultiServerMCPClient(
    {
        "sql": {
            "transport": "streamable_http",
            "url": SQL_SERVER_URL,
        }
    }
)

# Ask the server for its tools; each MCP tool comes back as a LangChain tool.
tools = await client.get_tools()
print("Tools from the MCP server:", [tool.name for tool in tools])

prebuilt_agent = create_agent(
    model="openai:gpt-4.1-mini",
    system_prompt=AGENT_INSTRUCTIONS,
    tools=tools,
)


Tools from the MCP server: ['list_tables', 'get_table_schema', 'execute_sql']


In [8]:
questions = [
    "How many customers pay in EUR?",
    "What is the average salary in each department?",
    "Which products were ordered by customers in Germany?",
]

for i, q in enumerate(questions, 1):
    print(f"\n────────── Test {i} ──────────")
    print(f"❓ {q}")
    result = await prebuilt_agent.ainvoke({"messages": [{"role": "user", "content": q}]})
    print(f"💬 {result['messages'][-1].content}")



────────── Test 1 ──────────
❓ How many customers pay in EUR?
💬 There are 4 customers who pay in EUR.

────────── Test 2 ──────────
❓ What is the average salary in each department?
💬 The average salary in each department is as follows:
- Engineering: 95,500.0
- Marketing: 70,000.0
- Sales: 60,000.0

────────── Test 3 ──────────
❓ Which products were ordered by customers in Germany?
💬 The products that were ordered by customers in Germany are: Widget A, Widget B, and Gadget Y.
